In [1]:


import pandas as pd
import plotly.express as px

import streamlit as st
from pathlib import Path
import re

import sys
# display columns and rows settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [10]:
Main_file_path = Path.cwd().parent
Data_file_path = Main_file_path / 'data'
#Data_file = Data_file_path / 'Basic_Macro_Indicators.csv'
Data_file = Data_file_path / 'Basic_Macro_Indicators_Cleaned.csv'

In [11]:
#DIR_DATA =  Path.cwd().parents[1].joinpath("DSCBI_PROJECT", "data")
#FILE = DIR_DATA.joinpath("Basic_Macro_Indicators.csv")
# ==============================
def load_data(Data_file=Data_file):
    data = pd.read_csv(Data_file)
    return data
# ==============================
data = load_data()


In [ ]:
sectors = list(data["sector"].unique())

sectors = list([n for n in sectors if n not in ['Debt sector','Other memo items']])
subsectors = list(
    data[data["sector"].isin(sectors)]["subsector"]
)

components = data[
    data["sector"].isin(sectors) &
    data["subsector"].isin(subsectors)
]["component"].unique().tolist()

units = data[
    data["sector"].isin(sectors) &
    data["subsector"].isin(subsectors) & data["component"].isin(components)
]["unit"].unique().tolist()

years = data["year"].unique().tolist()
Actual = [n for n in years if n <= 2025]
projection = [n for n in years if n > 2025]


In [69]:
value = data.query('sector =="Real Sector" and subsector== "GDP and components" and component=="Real GDP" and unit == "Billion RWF, Level" and year==2025')["value"]

#value = metric["value"]
#print(value.2f)


In [70]:
if not value.empty:
    val = value.squeeze()
    print(val)
else:
    val = None  # or any default value you want
    print("No matching data found")

13371.0


In [13]:
def get_value(df, sector, subsector, component, unit, year):

    filtered = data.query(
        'sector == @sector and subsector == @subsector and component == @component and unit == @unit and year == @year'
    )["value"]

    if not filtered.empty:
        return filtered.squeeze()  # returns scalar if single value
    else:
        return None


In [1]:
import pandas as pd
df = pd.read_excel(r"C:\Users\andrew.mushokambere\Downloads\Data.xlsx",sheet_name="Data") 


In [ ]:
df.head()

In [4]:
df_long = pd.melt(df, id_vars=["Series Name", "Country Name"], var_name="Year", value_name="Value")

In [ ]:
df_long.dropna

In [ ]:
df_long.Value.replace("..", 0)

In [14]:
df_long.to_csv(r"C:\Users\andrew.mushokambere\Downloads\Data1.csv",index=False)

In [ ]:
data.query('sector =="Real Sector" and subsector== "GDP and components" and component=="Real GDP"')
data.query('sector =="Real Sector" and subsector== "GDP and components" and component=="Real GDP"')

In [14]:
val = get_value(
    data, 
    sector="Fiscal sector", 
    subsector="Revenue", 
    component="Revenues incl. grants", 
    unit="Percentage of GDP",
    year=2025
)	

print(val)  # prints the value or None if not found


19.4


In [ ]:
data[data["sector"]=="Monetary sector"]

In [ ]:
#data.query('sector=="Real Sector" and subsector=="GDP and components" and year==2025')
data.query('sector=="Real Sector" and 	subsector=="Inflation and other prices" and year==2025')

data.query('sector=="Real Sector" ')

In [87]:
sectors

['Real Sector', 'External sector', 'Fiscal sector', 'Monetary sector']

In [46]:
#drop all unnamed columns
#data = data.loc[:, ~data.columns.str.contains('^Unnamed')]
#melting the dataframe
data_melted = data.melt(id_vars=['Sector', 'Subsector', 'Component'], var_name='Year', value_name='Value')


In [47]:
# searching and extracting everything between parentheses in a Component column and assign it to a new column 'Unit' and deleting it from Component column
data_melted['Unit'] = data_melted['Component'].str.extract(r'\((.*?)\)')
data_melted['Component'] = data_melted['Component'].str.replace(r'\s*\([^)]*\)', '', regex=True)

In [48]:
# bringing a Unit column next to Component column
cols = data_melted.columns.tolist()
cols.insert(3, cols.pop(cols.index('Unit')))
data_melted = data_melted[cols]
data_melted.head(3)

,Sector,Subsector,Component,Unit,Year,Value
0,Real Sector,GDP and components,Nominal GDP,"Billion RWF, Level",2011,"4,084"
1,Real Sector,GDP and components,Nominal GDP,"Million USD, Level",2011,"6,803"
2,Real Sector,GDP and components,Real GDP,"Billion RWF, Level",2011,"5,314"


In [49]:
# cleaning and removing spaces, commas and % in Value column
data_melted['Value'] = data_melted['Value'].str.replace('[%,]', '', regex=True).str.strip()




In [50]:
#checking the data types and converting Year and Value columns to appropriate data types
data_melted.dtypes
data_melted['Year'] = data_melted['Year'].astype(int)
data_melted['Value'] = pd.to_numeric(data_melted['Value'], errors='coerce')
data_melted.dtypes

Sector        object
Subsector     object
Component     object
Unit          object
Year           int64
Value        float64
dtype: object

In [55]:
#creating a new column colled Datatype and fill it with "Actual" where year is less than or equal to 2025 and "Projection" where year is greater than 2025
data_melted['Datatype'] = data_melted['Year'].apply(lambda x: 'Actual' if x <= 2025 else 'Projection')

In [59]:
#export the cleaned data to a new csv file
data_melted.to_csv(Data_file_path / 'Basic_Macro_Indicators_Cleaned.csv', index=False)

In [ ]:
import feedparser
import pandas as pd
from datetime import datetime

# ---------------------------
# RSS SOURCES
# ---------------------------
SOURCES = {
    "Reuters": "https://www.reuters.com/rssFeed/businessNews",
    "BBC Business": "https://feeds.bbci.co.uk/news/business/rss.xml",
    "CNBC Economy": "https://www.cnbc.com/id/20910258/device/rss/rss.html",
    "Yahoo Finance Economy": "https://finance.yahoo.com/rss/economy",
    "World Bank": "https://www.worldbank.org/en/news/all/rss",

    # Africa
    "AllAfrica Business": "https://allafrica.com/tools/headlines/rdf/business/headlines.rdf",
    "Africa Business+": "https://african.business/feed/",
    "The Africa Report": "https://www.theafricareport.com/feed/",
    # East Africa
    "The East African": "https://www.theeastafrican.co.ke/tea/business?view=xml",

    # Rwanda
    "New Times Rwanda": "https://www.newtimes.co.rw/rss/business",
    "MINECOFIN Rwanda": "https://www.minecofin.gov.rw/news/rss"
}


# ---------------------------
# FETCH NEWS
# ---------------------------
all_articles = []

for source, url in SOURCES.items():
    feed = feedparser.parse(url)
    for entry in feed.entries[:10]:  # limit per source
        all_articles.append({
            "Source": source,
            "Title": entry.get("title", ""),
            "Summary": entry.get("summary", ""),
            "Link": entry.get("link", ""),
            "Published": entry.get("published", ""),
            "Fetched_On": datetime.now().strftime("%Y-%m-%d %H:%M")
        })

# ---------------------------
# CREATE DATAFRAME
# ---------------------------
df = pd.DataFrame(all_articles)

# Sort by source then recency (if available)
df = df.sort_values(by=["Source"])

print(df.head())

# ---------------------------
# SAVE OUTPUT
# ---------------------------
df.to_csv("latest_economic_news.csv", index=False)


In [ ]:
import feedparser
import pandas as pd
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer

# ------------------------------------------------
# NEWS SOURCES (FREE & RELIABLE)
# ------------------------------------------------
SOURCES = [
    # -------- GLOBAL --------
    {"source": "Reuters", "region": "Global",
     "url": "https://www.reuters.com/rssFeed/businessNews"},

    {"source": "BBC Business", "region": "Global",
     "url": "https://feeds.bbci.co.uk/news/business/rss.xml"},

    {"source": "World Bank", "region": "Global",
     "url": "https://www.worldbank.org/en/news/all/rss"},

    {"source": "Yahoo Finance Economy", "region": "Global",
     "url": "https://finance.yahoo.com/rss/economy"},

    # -------- AFRICA --------
    {"source": "AllAfrica Business", "region": "Africa",
     "url": "https://allafrica.com/tools/headlines/rdf/business/headlines.rdf"},

    {"source": "Africa Business+", "region": "Africa",
     "url": "https://african.business/feed/"},

    # -------- EAST AFRICA --------
    {"source": "The East African", "region": "East Africa",
     "url": "https://www.theeastafrican.co.ke/tea/business?view=xml"},

    # -------- RWANDA --------
    {"source": "New Times Rwanda", "region": "Rwanda",
     "url": "https://www.newtimes.co.rw/rss/business"},

    {"source": "MINECOFIN Rwanda", "region": "Rwanda",
     "url": "https://www.minecofin.gov.rw/news/rss"},

    # Rwanda Broadcasting Agency (best-effort)
    {"source": "RBA Rwanda", "region": "Rwanda",
     "url": "https://www.rba.co.rw/rss"}  # may fail gracefully
]

# ------------------------------------------------
# FETCH NEWS SAFELY
# ------------------------------------------------
articles = []

for s in SOURCES:
    try:
        feed = feedparser.parse(s["url"])

        if not feed.entries:
            print(f"⚠️ No entries for {s['source']}")
            continue

        for entry in feed.entries[:10]:
            articles.append({
                "Region": s["region"],
                "Source": s["source"],
                "Title": entry.get("title", ""),
                "Summary": entry.get("summary", ""),
                "Link": entry.get("link", ""),
                "Published": entry.get("published", ""),
                "Fetched_On": datetime.now().strftime("%Y-%m-%d %H:%M")
            })

    except Exception as e:
        print(f"❌ Skipped {s['source']}: {e}")

df = pd.DataFrame(articles)
df.drop_duplicates(subset=["Title"], inplace=True)


In [13]:
def get_trending_terms(df, region, top_n=6):
    texts = (
        df[df["Region"] == region]["Title"]
        + " "
        + df[df["Region"] == region]["Summary"]
    )

    if texts.empty:
        return []

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=top_n
    )

    X = vectorizer.fit_transform(texts)
    return vectorizer.get_feature_names_out()


regions = df["Region"].unique()

trend_summary = []
for r in regions:
    trend_summary.append({
        "Region": r,
        "Trending Topics": ", ".join(get_trending_terms(df, r))
    })

trends_df = pd.DataFrame(trend_summary)


In [14]:
df.to_csv("economic_news_global_africa_eac_rwanda.csv", index=False)
trends_df.to_csv("trending_topics_by_region.csv", index=False)


In [10]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

# ==========================================
# 1️⃣ NEWS SOURCES (FREE)
# ==========================================
SOURCES = [
    # -------- GLOBAL --------
    {"source": "Reuters", "region": "Global",
     "url": "https://www.reuters.com/rssFeed/businessNews"},

    {"source": "BBC Business", "region": "Global",
     "url": "https://feeds.bbci.co.uk/news/business/rss.xml"},

    {"source": "World Bank", "region": "Global",
     "url": "https://www.worldbank.org/en/news/all/rss"},

    {"source": "Yahoo Finance Economy", "region": "Global",
     "url": "https://finance.yahoo.com/rss/economy"},

    # -------- AFRICA --------
    {"source": "AllAfrica Business", "region": "Africa",
     "url": "https://allafrica.com/tools/headlines/rdf/business/headlines.rdf"},

    {"source": "Africa Business+", "region": "Africa",
     "url": "https://african.business/feed/"},

    # -------- EAST AFRICA --------
    {"source": "The East African", "region": "East Africa",
     "url": "https://www.theeastafrican.co.ke/tea/business?view=xml"},

    # -------- RWANDA --------
    {"source": "New Times Rwanda", "region": "Rwanda",
     "url": "https://www.newtimes.co.rw/rss/business"},

    {"source": "MINECOFIN Rwanda", "region": "Rwanda",
     "url": "https://www.minecofin.gov.rw/news/rss"},

    # Rwanda Broadcasting Agency (best-effort)
    {"source": "RBA Rwanda", "region": "Rwanda",
     "url": "https://www.rba.co.rw/rss"}  # may fail gracefully
]

# ==========================================
# 2️⃣ FETCH NEWS
# ==========================================
articles = []

for s in SOURCES:
    feed = feedparser.parse(s["url"])
    for entry in feed.entries[:30]:
        articles.append({
            "Region": s["region"],
            "Source": s["source"],
            "Title": entry.get("title", ""),
            "Summary": entry.get("summary", ""),
            "Link": entry.get("link", ""),
        })

df = pd.DataFrame(articles).drop_duplicates(subset=["Title"])

# ==========================================
# 3️⃣ TRENDING TERMS (TF-IDF)
# ==========================================
def trending_terms(data, region, top_n=3):
    texts = data[data["Region"] == region]["Title"] + " " + data[data["Region"] == region]["Summary"]
    if texts.empty:
        return []
    tfidf = TfidfVectorizer(stop_words="english", max_features=top_n)
    tfidf.fit_transform(texts)
    return tfidf.get_feature_names_out()

regions = df["Region"].unique()
region_trends = {r: trending_terms(df, r) for r in regions}

# ==========================================
# 4️⃣ TOP HEADLINES
# ==========================================
top_headlines = {
    r: df[df["Region"] == r][["Source", "Title", "Summary", "Link"]].head(20)
    for r in regions
}

# ==========================================
# 5️⃣ EXPORT TO WORD
# ==========================================
doc = Document()
doc.add_heading("Automated Economic Brief", 0)
doc.add_paragraph(f"Generated on: {(datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')}")

for r in regions:
    doc.add_heading(r, level=1)

    doc.add_paragraph("Trending Economic Topics:")
    for term in region_trends[r]:
        doc.add_paragraph(term, style="List Bullet")

    doc.add_paragraph("Top Headlines:")
    for _, row in top_headlines[r].iterrows():
        doc.add_paragraph(f"Source: {row['Source']}")
        title = doc.add_paragraph()
        title.add_run(row["Title"]).bold = True
        doc.add_paragraph(row["Summary"])
        link = doc.add_paragraph()
        link.add_run(row["Link"]).italic = True
        doc.add_paragraph("")

doc.save("Automated_Economic_Brief.docx")

# ==========================================
# 6️⃣ EXPORT TO PDF
# ==========================================
pdf = SimpleDocTemplate("Automated_Economic_Brief.pdf", pagesize=A4)
styles = getSampleStyleSheet()
content = []

content.append(Paragraph("Automated Economic Brief", styles["Title"]))
content.append(Paragraph(
    f"Generated on: {(datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')}",
    styles["Normal"]
))

for r in regions:
    content.append(Paragraph(r, styles["Heading2"]))

    content.append(Paragraph("Trending Economic Topics:", styles["Heading3"]))
    for term in region_trends[r]:
        content.append(Paragraph(term, styles["Normal"]))

    content.append(Paragraph("Top Headlines:", styles["Heading3"]))
    for _, row in top_headlines[r].iterrows():
        #content.append(Paragraph(f"<b>Source:</b> {row['Source']}", styles["Normal"]))
        content.append(Paragraph(f"<b>{row['Title']}</b>", styles["Heading3"]))
        content.append(Paragraph(row["Summary"], styles["Normal"]))
        content.append(Paragraph(row["Link"], styles["Italic"]))
        content.append(Paragraph("<br/>", styles["Normal"]))

pdf.build(content)

print("✅ Word and PDF economic brief generated successfully!")


✅ Word and PDF economic brief generated successfully!


# **WITH REPORTING DATE**

In [1]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

# ==========================================
# 1️⃣ NEWS SOURCES
# ==========================================
# SOURCES = [
#     {"source": "Reuters", "region": "Global",
#      "url": "https://www.reuters.com/rssFeed/businessNews"},
#     {"source": "BBC Business", "region": "Global",
#      "url": "https://feeds.bbci.co.uk/news/business/rss.xml"},
#     {"source": "World Bank", "region": "Global",
#      "url": "https://www.worldbank.org/en/news/all/rss"},
#     {"source": "Yahoo Finance Economy", "region": "Global",
#      "url": "https://finance.yahoo.com/rss/economy"},
#     {"source": "AllAfrica Business", "region": "Africa",
#      "url": "https://allafrica.com/tools/headlines/rdf/business/headlines.rdf"},
#     {"source": "Africa Business+", "region": "Africa",
#      "url": "https://african.business/feed/"},
#     {"source": "The East African", "region": "East Africa",
#      "url": "https://www.theeastafrican.co.ke/tea/business?view=xml"},
#     {"source": "New Times Rwanda", "region": "Rwanda",
#      "url": "https://www.newtimes.co.rw/rss/business"},
#     {"source": "MINECOFIN Rwanda", "region": "Rwanda",
#      "url": "https://www.minecofin.gov.rw/news/rss"},
#     {"source": "RBA Rwanda", "region": "Rwanda",
#      "url": "https://www.rba.co.rw/rss"}
# ]

SOURCES = [

    # ==========================
    # GLOBAL ECONOMY & FINANCE
    # ==========================

    {"source": "Reuters Business", "region": "Global",
     "url": "https://www.reuters.com/rssFeed/businessNews"},

    {"source": "Reuters World", "region": "Global",
     "url": "https://www.reuters.com/rssFeed/worldNews"},

    {"source": "BBC Business", "region": "Global",
     "url": "https://feeds.bbci.co.uk/news/business/rss.xml"},

    {"source": "BBC World", "region": "Global",
     "url": "https://feeds.bbci.co.uk/news/world/rss.xml"},

    {"source": "World Bank News", "region": "Global",
     "url": "https://www.worldbank.org/en/news/all/rss"},

    {"source": "IMF News", "region": "Global",
     "url": "https://www.imf.org/external/rss/rss.aspx"},

    {"source": "OECD Economy", "region": "Global",
     "url": "https://www.oecd.org/newsroom/index.xml"},

    {"source": "Yahoo Finance Economy", "region": "Global",
     "url": "https://finance.yahoo.com/rss/economy"},

    {"source": "Financial Times Global Economy", "region": "Global",
     "url": "https://www.ft.com/global-economy?format=rss"},

    {"source": "Bloomberg Economics", "region": "Global",
     "url": "https://www.bloomberg.com/feed/podcast/etf-report.xml"},


    # ==========================
    # AFRICA CONTINENTAL NEWS
    # ==========================

    {"source": "AllAfrica Business", "region": "Africa",
     "url": "https://allafrica.com/tools/headlines/rdf/business/headlines.rdf"},

    {"source": "AllAfrica Politics", "region": "Africa",
     "url": "https://allafrica.com/tools/headlines/rdf/politics/headlines.rdf"},

    {"source": "African Business Magazine", "region": "Africa",
     "url": "https://african.business/feed/"},

    {"source": "Africa Report", "region": "Africa",
     "url": "https://www.theafricareport.com/feed/"},

    {"source": "Africa News", "region": "Africa",
     "url": "https://www.africanews.com/feed/"},

    {"source": "UNECA News", "region": "Africa",
     "url": "https://www.uneca.org/rss.xml"},


    # ==========================
    # EAST AFRICA REGIONAL
    # ==========================

    {"source": "The East African - Business", "region": "East Africa",
     "url": "https://www.theeastafrican.co.ke/tea/business?view=xml"},

    {"source": "The East African - Politics", "region": "East Africa",
     "url": "https://www.theeastafrican.co.ke/tea/news/east-africa?view=xml"},

    {"source": "Daily Nation Kenya Business", "region": "East Africa",
     "url": "https://nation.africa/rss/business"},

    {"source": "Daily Nation Kenya Politics", "region": "East Africa",
     "url": "https://nation.africa/rss/politics"},

    {"source": "Monitor Uganda Business", "region": "East Africa",
     "url": "https://www.monitor.co.ug/business/rss"},

    {"source": "Monitor Uganda Politics", "region": "East Africa",
     "url": "https://www.monitor.co.ug/uganda/news/national/rss"},

    {"source": "Citizen Tanzania", "region": "East Africa",
     "url": "https://www.thecitizen.co.tz/rss"},

    {"source": "EAC Secretariat News", "region": "East Africa",
     "url": "https://www.eac.int/rss"},


    # ==========================
    # RWANDA – LOCAL ECONOMIC & POLITICAL
    # ==========================

    {"source": "New Times Rwanda - Business", "region": "Rwanda",
     "url": "https://www.newtimes.co.rw/rss/business"},

    {"source": "New Times Rwanda - Politics", "region": "Rwanda",
     "url": "https://www.newtimes.co.rw/rss/politics"},

    {"source": "Rwanda Broadcasting Agency (RBA)", "region": "Rwanda",
     "url": "https://www.rba.co.rw/rss"},

    {"source": "MINECOFIN Rwanda", "region": "Rwanda",
     "url": "https://www.minecofin.gov.rw/news/rss"},

    {"source": "BNR – National Bank of Rwanda News", "region": "Rwanda",
     "url": "https://www.bnr.rw/rss"},

    {"source": "Rwanda Development Board (RDB)", "region": "Rwanda",
     "url": "https://rdb.rw/feed/"},

    {"source": "Rwanda Today", "region": "Rwanda",
     "url": "https://www.rwandatoday.africa/rss"},

    {"source": "KT Press Rwanda", "region": "Rwanda",
     "url": "https://www.ktpress.rw/feed/"},

    {"source": "Igihe Rwanda (EN)", "region": "Rwanda",
     "url": "https://en.igihe.com/spip.php?page=backend"},

    {"source": "Rwanda Parliament News", "region": "Rwanda",
     "url": "https://www.parliament.gov.rw/rss"},

]


# ==========================================
# 2️⃣ FETCH NEWS
# ==========================================
articles = []

for s in SOURCES:
    feed = feedparser.parse(s["url"])
    for entry in feed.entries[:30]:

        # Safely extract reporting date
        published = entry.get("published", entry.get("updated", ""))
        try:
            reporting_date = datetime(*entry.published_parsed[:6]).strftime("%Y-%m-%d")
        except:
            reporting_date = "N/A"

        articles.append({
            "Region": s["region"],
            "Source": s["source"],
            "Title": entry.get("title", ""),
            "Summary": entry.get("summary", ""),
            "Link": entry.get("link", ""),
            "Reporting_Date": reporting_date
        })

df = pd.DataFrame(articles).drop_duplicates(subset=["Title"])

# ==========================================
# 3️⃣ TRENDING TERMS (TF-IDF)
# ==========================================
def trending_terms(data, region, top_n=3):
    texts = data[data["Region"] == region]["Title"] + " " + data[data["Region"] == region]["Summary"]
    if texts.empty:
        return []
    tfidf = TfidfVectorizer(stop_words="english", max_features=top_n)
    tfidf.fit_transform(texts)
    return tfidf.get_feature_names_out()

regions = df["Region"].unique()
region_trends = {r: trending_terms(df, r) for r in regions}

# ==========================================
# 4️⃣ TOP HEADLINES
# ==========================================
top_headlines = {
    r: df[df["Region"] == r][
        ["Source", "Title", "Summary", "Link", "Reporting_Date"]
    ].head(20)
    for r in regions
}

# ==========================================
# 5️⃣ EXPORT TO WORD
# ==========================================
doc = Document()
doc.add_heading("Automated Economic Brief", 0)
doc.add_paragraph(f"Generated on: {datetime.now().strftime('%Y-%m-%d')}")

for r in regions:
    doc.add_heading(r, level=1)

    doc.add_paragraph("Trending Economic Topics:")
    for term in region_trends[r]:
        doc.add_paragraph(term, style="List Bullet")

    doc.add_paragraph("Top Headlines:")
    for _, row in top_headlines[r].iterrows():
        doc.add_paragraph(f"Source: {row['Source']}")
        title = doc.add_paragraph()
        title.add_run(row["Title"]).bold = True
        doc.add_paragraph(row["Summary"])
        doc.add_paragraph(f"Reporting date: {row['Reporting_Date']}", style="Intense Quote")
        link = doc.add_paragraph()
        link.add_run(row["Link"]).italic = True
        doc.add_paragraph("")

doc.save("Automated_Economic_Brief.docx")

# ==========================================
# 6️⃣ EXPORT TO PDF
# ==========================================
pdf = SimpleDocTemplate("Automated_Economic_Brief.pdf", pagesize=A4)
styles = getSampleStyleSheet()
content = []

content.append(Paragraph("Automated Economic Brief", styles["Title"]))
content.append(Paragraph(
    f"Generated on: {datetime.now().strftime('%Y-%m-%d')}",
    styles["Normal"]
))

for r in regions:
    content.append(Paragraph(r, styles["Heading2"]))

    content.append(Paragraph("Trending Economic Topics:", styles["Heading3"]))
    for term in region_trends[r]:
        content.append(Paragraph(term, styles["Normal"]))

    content.append(Paragraph("Top Headlines:", styles["Heading3"]))
    for _, row in top_headlines[r].iterrows():
        content.append(Paragraph(f"<b>{row['Title']}</b>", styles["Heading3"]))
        content.append(Paragraph(row["Summary"], styles["Normal"]))
        content.append(Paragraph(
            f"<i>Reporting date: {row['Reporting_Date']}</i>",
            styles["Italic"]
        ))
        content.append(Paragraph(row["Link"], styles["Italic"]))
        content.append(Paragraph("<br/>", styles["Normal"]))

pdf.build(content)

print("✅ Word and PDF economic brief generated successfully!")


✅ Word and PDF economic brief generated successfully!


In [4]:
# a list of months and years from Jan-09 to Dec-25
Period = [
    "Jan-09", "Feb-09", "Mar-09", "Apr-09", "May-09", "Jun-09",
    "Jul-09", "Aug-09", "Sep-09", "Oct-09", "Nov-09", "Dec-09",
    "Jan-10", "Feb-10", "Mar-10", "Apr-10", "May-10", "Jun-10",
    "Jul-10", "Aug-10", "Sep-10", "Oct-10", "Nov-10", "Dec-10",
    "Jan-11", "Feb-11", "Mar-11", "Apr-11", "May-11", "Jun-11",
    "Jul-11", "Aug-11", "Sep-11", "Oct-11", "Nov-11", "Dec-11",
    "Jan-12", "Feb-12", "Mar-12", "Apr-12", "May-12", 	"Jun-12", "Jul-12", "Aug-12", "Sep-12", "Oct-12", "Nov-12", "Dec-12",
    "Jan-13", "Feb-13", "Mar-13", "Apr-13", "May-13", 	"Jun-13", "Jul-13", "Aug-13", "Sep-13", "Oct-13", "Nov-13", "Dec-13",
    "Jan-14", "Feb-14", "Mar-14", "Apr-14", "May-14", 	"Jun-14", "Jul-14", "Aug-14", "Sep-14", "Oct-14", "Nov-14", "Dec-14",
    "Jan-15", "Feb-15", "Mar-15", "Apr-15", "May-15", 	"Jun-15", "Jul-15", "Aug-15", "Sep-15", "Oct-15", "Nov-15", "Dec-15",
    "Jan-16", "Feb-16", "Mar-16", "Apr-16", "May-16", 	"Jun-16", "Jul-16", "Aug-16", "Sep-16", "Oct-16", "Nov-16", "Dec-16",
    "Jan-17", "Feb-17", "Mar-17", "Apr-17", "May-17", 	"Jun-17", "Jul-17", "Aug-17", "Sep-17", "Oct-17", "Nov-17", "Dec-17",
    "Jan-18", "Feb-18", "Mar-18", "Apr-18", "May-18", 	"Jun_25", "Jul-18", "Aug-18", "Sep-18", "Oct-18", "Nov-18", "Dec-18",
    "Jan-19", "Feb-19", "Mar-19", "Apr-19", "May-19", 	"Jun-19", "Jul-19", "Aug-19", "Sep-19", "Oct-19", "Nov-19", "Dec-19",
    "Jan-20", "Feb-20", "Mar-20", "Apr-20", "May-20", 	"Jun-20", "Jul-20", "Aug-20", "Sep-20", "Oct-20", "Nov-20", "Dec-20",
    "Jan-21", "Feb-21", "Mar-21", "Apr-21", "May-21", 	"Jun-21", "Jul-21", "Aug-21", "Sep-21", "Oct-21", "Nov-21", "Dec-21",
    "Jan-22", "Feb-22", "Mar-22", "Apr-22", "May-22", 	"Jun-22", "Jul-22", "Aug-22", "Sep-22", "Oct-22", "Nov-22", "Dec-22",
    "Jan-23", "Feb-23", "Mar-23", "Apr-23", "May-23", 	"Jun-23", "Jul-23", "Aug-23", "Sep-23", "Oct-23", "Nov-23", "Dec-23",
    "Jan-24", "Feb-24", "Mar-24", "Apr-24", "May-24", 	"Jun-24", "Jul-24", "Aug-24", "Sep-24", "Oct-24", "Nov-24", "Dec-24",
    "Jan-25", "Feb-25", "Mar-25", "Apr-25", "May-25", 	"Jun-25", "Jul-25", "Aug-25", "Sep-25", "Oct-25", "Nov-25", "Dec-25"    
]


In [3]:
file = pd.read_csv("E:\\Documents\\MINECOFIN\\Projects\\US-IRAN OIL IMPACT ANALYSIS\\Book1.csv")

In [17]:
# converting the first column to datetime and extracting month and year
#file["Date"] = pd.to_datetime(file["period"], errors="coerce")
file["Month"] = file["date"].dt.month
file["Year"] = file["date"].dt.year

# # generate a range of months and years from Jan-09 to Dec-25 and convert it to datetime
# date_range = pd.date_range(start="2009-01-01", end="2025-12-31", freq="MS")
# date_range_str = date_range.strftime("%b-%y").tolist()

In [13]:
file["date"] = date_range

In [16]:
file.drop(columns=["period", "Date"], inplace=True)

In [23]:
# aggregating the data by year by taking the mean of price_index for each 12 months in a year

monthly_data = file.groupby(["Year", "Month"])["price_index"].mean().reset_index()

annual_data = file.groupby(["Year"])["price_index"].mean().reset_index()

In [26]:
annual_data.to_csv("annual_price_index.csv", index=False)

In [20]:
file.head()

,price_index,Month,Year,date
0,80.53,1,2009,2009-01-01
1,81.39,2,2009,2009-02-01
2,82.07,3,2009,2009-03-01
3,81.93,4,2009,2009-04-01
4,81.21,5,2009,2009-05-01
